# Q1 — Runtime commitment and real drop-off

**Question.** Among viewers who have actually started an anime, how are episode count and total runtime associated with dropping, early quitting, and the TV one-cour versus two-cour decision?

**Decision.** Production and streaming teams can use this evidence to decide when a compact split-cour structure or earlier engagement intervention deserves testing. This is observational evidence, not causal proof.

## Storyboard

1. Explain why Plan to Watch is excluded from the denominator.
2. Show active drop-off across episode bands.
3. Compare one-cour and two-cour TV series.
4. Check the comparison in major genres.
5. Show how early recorded dropping happens.

In [ ]:
from __future__ import annotations
from collections import Counter
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.audit import report_row_change
from src.utils import bootstrap_ci, iter_list_parts, load_clean_tables, setup_plot_style

SEED, MIN_ACTIVE_STARTERS = 42, 100
OUT_DIR = ROOT / 'artifacts' / 'q1'
OUT_DIR.mkdir(parents=True, exist_ok=True)
setup_plot_style()
print(f'Project: {ROOT}\nChart folder: {OUT_DIR}')

## Load and aggregate the cleaned data

The user–anime table has about 80M rows. This cell streams its Parquet fragments, retaining only the counts and early-drop measures needed for Q1.

In [ ]:
anime, users, parts, logs = load_clean_tables(ROOT)
episodes = anime.set_index('anime_id')['episodes_clean']
by_status = {s: Counter() for s in [1, 2, 3, 4, 6]}
all_status, drop_n, early_n, progress_sum = Counter(), Counter(), Counter(), Counter()
for i, d in iter_list_parts(parts, columns=['anime_id','my_status_clean','my_watched_episodes_clean']):
    d = d.dropna(subset=['my_status_clean']).copy(); d['my_status_clean'] = d['my_status_clean'].astype(int)
    d = d[d.my_status_clean.isin(by_status)]
    all_status.update(d.my_status_clean.value_counts().to_dict())
    for s in by_status: by_status[s].update(d.loc[d.my_status_clean.eq(s), 'anime_id'].value_counts().to_dict())
    x = d[d.my_status_clean.eq(4)].dropna(subset=['my_watched_episodes_clean']).copy()
    x['episodes'] = x.anime_id.map(episodes); x = x[x.episodes.gt(0)]
    x['progress'] = x.my_watched_episodes_clean / x.episodes; x = x[x.progress.between(0, 1)]
    drop_n.update(x.anime_id.value_counts().to_dict())
    early_n.update(x.assign(early=x.progress.le(.25)).groupby('anime_id').early.sum().astype(int).to_dict())
    progress_sum.update(x.groupby('anime_id').progress.sum().to_dict())
    print(f'Part {i}/{len(parts)}', end='\r')
print('\nAggregation complete.')

agg = pd.DataFrame({**{f's{s}': pd.Series(c, dtype=float) for s,c in by_status.items()}, 'drop_n':pd.Series(drop_n,dtype=float), 'early_n':pd.Series(early_n,dtype=float), 'progress_sum':pd.Series(progress_sum,dtype=float)}).fillna(0)
agg['active_starters'] = agg[['s1','s2','s3','s4']].sum(axis=1)
agg['active_drop_rate'] = np.where(agg.active_starters.gt(0), agg.s4 / agg.active_starters, np.nan)
agg['early25_drop_share'] = np.where(agg.drop_n.ge(20), agg.early_n / agg.drop_n, np.nan)
full = anime.merge(agg, left_on='anime_id', right_index=True, how='left').fillna(0)

## 1. Why Plan to Watch is not part of the Q1 denominator

A planned title has not been started, so it cannot yet lead to either completion or dropping. Including it would artificially lower the apparent drop rate.

In [ ]:
labels = {1:'Watching',2:'Completed',3:'On hold',4:'Dropped',6:'Plan to Watch'}
status = pd.DataFrame({'code':list(all_status), 'rows':list(all_status.values())})
status['status'] = status.code.map(labels); status['share (%)'] = 100*status.rows/status.rows.sum(); status = status.sort_values('rows', ascending=False)
display(status[['status','rows','share (%)']])
fig, ax = plt.subplots(figsize=(8,4.5)); sns.barplot(status, y='status', x='share (%)', hue='status', palette='viridis', legend=False, ax=ax)
for y, r in enumerate(status.itertuples()): ax.text(r._4+.3, y, f'{r._4:.2f}%', va='center')
ax.set(title='Plan to Watch is large—but not an eligible Q1 outcome', ylabel=''); fig.tight_layout(); fig.savefig(OUT_DIR/'01_status_denominator.png'); plt.show()

## 2. Define the analytic sample

Use finished anime with known episode count, duration and format. Require at least 100 active starters so rates are not dominated by very small audiences.

In [ ]:
q0 = full.copy(); q1 = q0[q0.status_clean.eq('Finished Airing')].copy(); report_row_change('Q1: finished airing', q0, q1, 'stable viewing outcome')
q2 = q1[q1.episodes_clean.gt(0) & q1.duration_minutes_clean.gt(0) & q1.type_clean.notna()].copy(); report_row_change('Q1: usable metadata', q1, q2, 'measure commitment')
q = q2[q2.active_starters.ge(MIN_ACTIVE_STARTERS)].copy(); report_row_change('Q1: minimum audience', q2, q, 'stable anime-level rates')
q['runtime_hours'] = q.episodes_clean*q.duration_minutes_clean/60
q['episode_band'] = pd.cut(q.episodes_clean, [0,1,6,13,26,52,np.inf], labels=['1','2–6','7–13','14–26','27–52','53+'])
tv = q[q.type_clean.eq('TV')].copy(); tv['cour'] = np.select([tv.episodes_clean.between(10,13),tv.episodes_clean.between(22,26),tv.episodes_clean.ge(27)], ['1-cour (10–13)','2-cour (22–26)','Long-running (27+)'], default='Other TV')
print(f'Analytic sample: {len(q):,} anime; TV subset: {len(tv):,}')

## 3. Main pattern: episode commitment and dropping

Mean anime-level active drop rate, with bootstrap 95% confidence intervals. The denominator is `watching + completed + on_hold + dropped` only.

In [ ]:
rows=[]
for band, g in q.groupby('episode_band', observed=True):
    mean, lo, hi = bootstrap_ci(g.active_drop_rate*100, n_boot=1000, seed=SEED)
    rows.append({'Episode band':str(band),'Anime':len(g),'Drop rate (%)':mean,'CI low':lo,'CI high':hi,'Early quit (%)':100*g.early25_drop_share.mean()})
summary = pd.DataFrame(rows); display(summary)
fig, ax = plt.subplots(figsize=(9,4.8)); ax.errorbar(summary['Episode band'],summary['Drop rate (%)'],yerr=[summary['Drop rate (%)']-summary['CI low'],summary['CI high']-summary['Drop rate (%)']],marker='o',capsize=4,color='#b43c59')
ax.set(title='Active drop-off across episode-count bands',xlabel='Episodes',ylabel='Mean active drop rate (%)'); ax.set_ylim(bottom=0); fig.tight_layout(); fig.savefig(OUT_DIR/'02_episode_band_drop_rate.png'); plt.show()
rho,p = stats.spearmanr(tv.runtime_hours,tv.active_drop_rate,nan_policy='omit'); print(f'TV-only runtime correlation: Spearman r={rho:.3f}, p={p:.3g}')

## 4. Controlled TV comparison

Comparing only TV series removes the largest format difference. The result remains associational because genre, era and franchise context may still differ.

In [ ]:
order=['1-cour (10–13)','2-cour (22–26)','Long-running (27+)']; c = tv[tv.cour.isin(order)].copy()
cour = c.groupby('cour',observed=True).active_drop_rate.agg(['mean','count']).reindex(order).reset_index(); display(cour.assign(**{'drop rate (%)':cour['mean']*100}))
one=c.loc[c.cour.eq(order[0]),'active_drop_rate']*100; two=c.loc[c.cour.eq(order[1]),'active_drop_rate']*100; _,p=stats.mannwhitneyu(one,two,alternative='two-sided')
print(f'One-cour {one.mean():.2f}% vs two-cour {two.mean():.2f}% (Mann–Whitney p={p:.3g})')
fig,ax=plt.subplots(figsize=(8,4.8)); sns.barplot(cour,x='cour',y='mean',hue='cour',palette='magma',legend=False,ax=ax); ax.set(title='TV: season structure and drop-off',xlabel='',ylabel='Mean active drop rate'); ax.yaxis.set_major_formatter(lambda x,pos:f'{x*100:.0f}%'); fig.tight_layout(); fig.savefig(OUT_DIR/'03_tv_cour.png'); plt.show()

## 5. Genre robustness and early quitting

Genres overlap, so each panel is a within-genre check rather than mutually exclusive segmentation. Early quitting uses only titles with at least 20 recorded dropped rows.

In [ ]:
genres=['Action','Comedy','Drama','Fantasy','Romance','Sci-Fi']; gr=[]
for genre in genres:
    z=c[c.genre_clean.fillna('').str.contains(genre,regex=False)]
    for name,g in z.groupby('cour',observed=True):
        if name in order[:2] and len(g)>=10: gr.append({'Genre':genre,'Structure':name,'Drop rate (%)':100*g.active_drop_rate.mean(),'Anime':len(g)})
genre=pd.DataFrame(gr); display(genre)
fig,axs=plt.subplots(1,2,figsize=(14,4.8)); sns.barplot(genre,x='Genre',y='Drop rate (%)',hue='Structure',ax=axs[0]); axs[0].set(title='TV comparison within major genres',xlabel='')
early=q.groupby('episode_band',observed=True).early25_drop_share.mean().mul(100).reset_index(name='Early quit (%)'); sns.barplot(early,x='episode_band',y='Early quit (%)',hue='episode_band',palette='crest',legend=False,ax=axs[1]); axs[1].set(title='Recorded drop-outs leaving by 25% watched',xlabel='Episodes'); axs[1].set_ylim(0,100)
fig.tight_layout(); fig.savefig(OUT_DIR/'04_genres_and_early_drop.png'); plt.show()

## Conclusion and presentation notes

Present the figures in order: denominator → episode trend → TV decision → genre check → early quitting.

Report the executed values, then state the limits: cross-sectional, self-reported lists, unmeasured confounders, and no causal conclusion. The practical implication is to test split-cour design or early-engagement support for longer TV series—not to assume length alone causes dropping.